## Import libraries

In [1]:
import gc
# import wandb
import random
import datetime
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

import torchtext
from torchtext.data import Field, BucketIterator

##  Utils

In [2]:
def _tokenize(text):
    return [char for char in text]


def initialize_weights(m):
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight.data)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Transformer

### Enc, Enc Layer, Dec, Dec Layer, Pwise att, FF att

In [3]:
class Encoder(nn.Module):
    def __init__(self,
                 input_dim,
                 hid_dim,
                 n_layers,
                 n_heads,
                 pf_dim,
                 dropout,
                 device=torch.device("cuda"),
                 max_length=100):
        super().__init__()

        self.device = device
        self.tok_embedding = nn.Embedding(input_dim, hid_dim)
        self.pos_embedding = nn.Embedding(max_length, hid_dim)

        self.layers = nn.ModuleList([EncoderLayer(hid_dim, n_heads, pf_dim, dropout, device)
                                     for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([hid_dim])).to(device)

        self.hid_dim = hid_dim

    def forward(self, src, src_mask):
        # src = [batch size, src len]
        # src_mask = [batch size, src len]
        batch_size = src.shape[0]
        src_len = src.shape[1]

        pos = torch.arange(0, src_len).unsqueeze(
            0).repeat(batch_size, 1).to(self.device)
        # pos = [batch size, src len]

        src = self.dropout(
            (self.tok_embedding(src) * self.scale) + self.pos_embedding(pos))
        # self.tok_embedding(src) = [batch size, src len, hid dim]
        # self.pos_embedding(pos) = [batch size, src len, hid dim]
        # src = [batch size, src len, hid dim]

        for layer in self.layers:
            src = layer(src, src_mask)
        # src = [batch size, src len, hid dim]

        return src


class EncoderLayer(nn.Module):
    def __init__(self,
                 hid_dim,
                 n_heads,
                 pf_dim,
                 dropout,
                 device):
        super().__init__()

        self.self_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.ff_layer_norm = nn.LayerNorm(hid_dim)
        self.self_attention = MultiHeadAttentionLayer(
            hid_dim, n_heads, dropout, device)
        self.positionwise_feedforward = PositionwiseFeedforwardLayer(
            hid_dim, pf_dim, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask):
        # src = [batch size, src len, hid dim]
        # src_mask = [batch size, src len]

        # self attention
        _src, _ = self.self_attention(src, src, src, src_mask)

        # dropout, residual connection and layer norm
        src = self.self_attn_layer_norm(src + self.dropout(_src))
        # src = [batch size, src len, hid dim]

        # positionwise feedforward
        _src = self.positionwise_feedforward(src)

        # dropout, residual and layer norm
        src = self.ff_layer_norm(src + self.dropout(_src))
        # src = [batch size, src len, hid dim]

        return src


class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dim, n_heads, dropout, device):
        super().__init__()

        assert hid_dim % n_heads == 0

        self.hid_dim = hid_dim
        self.n_heads = n_heads
        self.head_dim = hid_dim // n_heads

        self.fc_q = nn.Linear(hid_dim, hid_dim)
        self.fc_k = nn.Linear(hid_dim, hid_dim)
        self.fc_v = nn.Linear(hid_dim, hid_dim)

        self.fc_o = nn.Linear(hid_dim, hid_dim)

        self.dropout = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([self.head_dim])).to(device)

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]
        # query = [batch size, query len, hid dim]
        # key = [batch size, key len, hid dim]
        # value = [batch size, value len, hid dim]

        Q = self.fc_q(query)
        K = self.fc_k(key)
        V = self.fc_v(value)
        # Q = [batch size, query len, hid dim]
        # K = [batch size, key len, hid dim]
        # V = [batch size, value len, hid dim]

        Q = Q.view(batch_size, -1, self.n_heads,
                   self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch_size, -1, self.n_heads,
                   self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch_size, -1, self.n_heads,
                   self.head_dim).permute(0, 2, 1, 3)
        # Q = [batch size, n heads, query len, head dim]
        # K = [batch size, n heads, key len, head dim]
        # V = [batch size, n heads, value len, head dim]

        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        # energy = [batch size, n heads, query len, key len]

        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)

        attention = torch.softmax(energy, dim=-1)
        # attention = [batch size, n heads, query len, key len]
        x = torch.matmul(self.dropout(attention), V)
        # x = [batch size, n heads, query len, head dim]
        x = x.permute(0, 2, 1, 3).contiguous()
        # x = [batch size, query len, n heads, head dim]
        x = x.view(batch_size, -1, self.hid_dim)
        # x = [batch size, query len, hid dim]
        x = self.fc_o(x)
        # x = [batch size, query len, hid dim]

        return x, attention


class PositionwiseFeedforwardLayer(nn.Module):
    def __init__(self, hid_dim, pf_dim, dropout):
        super().__init__()

        self.fc_1 = nn.Linear(hid_dim, pf_dim)
        self.fc_2 = nn.Linear(pf_dim, hid_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x = [batch size, seq len, hid dim]

        x = self.dropout(torch.relu(self.fc_1(x)))
        # x = [batch size, seq len, pf dim]

        x = self.fc_2(x)
        # x = [batch size, seq len, hid dim]

        return x


class Decoder(nn.Module):
    def __init__(self,
                 output_dim,
                 hid_dim,
                 n_layers,
                 n_heads,
                 pf_dim,
                 dropout,
                 device,
                 max_length=100):
        super().__init__()

        self.device = device

        self.tok_embedding = nn.Embedding(output_dim, hid_dim)
        self.pos_embedding = nn.Embedding(max_length, hid_dim)

        self.layers = nn.ModuleList([DecoderLayer(hid_dim, n_heads, pf_dim, dropout, device)
                                     for _ in range(n_layers)])

        self.fc_out = nn.Linear(hid_dim, output_dim)
        # self.fc_out = nn.Linear(hid_dim + 8, output_dim)

        self.dropout = nn.Dropout(dropout)

        self.scale = torch.sqrt(torch.FloatTensor([hid_dim])).to(device)

    def forward(self, trg, enc_src, trg_mask, src_mask):
        # trg = [batch size, trg len]
        # enc_src = [batch size, src len, hid dim]
        # trg_mask = [batch size, trg len]
        # src_mask = [batch size, src len]

        batch_size = trg.shape[0]
        trg_len = trg.shape[1]

        pos = torch.arange(0, trg_len).unsqueeze(
            0).repeat(batch_size, 1).to(self.device)
        # pos = [batch size, trg len]

        trg = self.dropout(
            (self.tok_embedding(trg) * self.scale) + self.pos_embedding(pos))
        # trg = [batch size, trg len, hid dim]

        for layer in self.layers:
            trg, attention = layer(trg, enc_src, trg_mask, src_mask)
        # trg = [batch size, trg len, hid dim]
        # attention = [batch size, n heads, trg len, src len]

        output = self.fc_out(trg)
        # output = [batch size, trg len, output dim]

        return output, attention


class DecoderLayer(nn.Module):
    def __init__(self,
                 hid_dim,
                 n_heads,
                 pf_dim,
                 dropout,
                 device):
        super().__init__()

        self.self_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.enc_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.ff_layer_norm = nn.LayerNorm(hid_dim)
        self.self_attention = MultiHeadAttentionLayer(
            hid_dim, n_heads, dropout, device)
        self.encoder_attention = MultiHeadAttentionLayer(
            hid_dim, n_heads, dropout, device)
        self.positionwise_feedforward = PositionwiseFeedforwardLayer(
            hid_dim, pf_dim, dropout)

        self.dropout = nn.Dropout(dropout)

    def forward(self, trg, enc_src, trg_mask, src_mask):
        # trg = [batch size, trg len, hid dim]
        # enc_src = [batch size, src len, hid dim]
        # trg_mask = [batch size, trg len]
        # src_mask = [batch size, src len]

        # self attention
        _trg, _ = self.self_attention(trg, trg, trg, trg_mask)

        # dropout, residual connection and layer norm
        trg = self.self_attn_layer_norm(trg + self.dropout(_trg))
        # trg = [batch size, trg len, hid dim]

        # encoder attention
        _trg, attention = self.encoder_attention(
            trg, enc_src, enc_src, src_mask)

        # dropout, residual connection and layer norm
        trg = self.enc_attn_layer_norm(trg + self.dropout(_trg))
        # trg = [batch size, trg len, hid dim]

        # positionwise feedforward
        _trg = self.positionwise_feedforward(trg)

        # dropout, residual and layer norm
        trg = self.ff_layer_norm(trg + self.dropout(_trg))
        # trg = [batch size, trg len, hid dim]
        # attention = [batch size, n heads, trg len, src len]

        return trg, attention


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\gMBA\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\gMBA\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\gMBA\venv\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "d:\gMBA\venv\Lib\site-packages\tornado\platform\asyncio.py", line 211, in 

### Seq2Seq

In [4]:
class Seq2Seq(nn.Module):

    def __init__(self,
                 encoder,
                 decoder,
                 src_pad_idx,
                 trg_pad_idx,
                 device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.truth_table_linear = nn.Linear(
            32, self.encoder.hid_dim, device=device)

        self.device = device

    def make_src_mask(self, src):
        # src = [batch size, src len]

        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        # src_mask = [batch size, 1, 1, src len]

        return src_mask

    def make_trg_mask(self, trg):
        # trg = [batch size, trg len]

        trg_pad_mask = (trg != self.trg_pad_idx).unsqueeze(1).unsqueeze(2)
        # trg_pad_mask = [batch size, 1, 1, trg len]

        trg_len = trg.shape[1]
        trg_sub_mask = torch.tril(torch.ones(
            (trg_len, trg_len), device=self.device)).bool()
        # trg_sub_mask = [trg len, trg len]

        trg_mask = trg_pad_mask & trg_sub_mask
        # trg_mask = [batch size, 1, trg len, trg len]

        return trg_mask

    def forward(self, src, trg, truth_table):
        # src = [batch size, src len]
        # trg = [batch size, trg len]

        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        # src_mask = [batch size, 1, 1, src len]
        # trg_mask = [batch size, 1, trg len, trg len]

        enc_src = self.encoder(src, src_mask)
        # enc_src = [batch size, src len, hid dim]

        truth_table_vector = self.truth_table_linear(truth_table)
        truth_table_vector = truth_table_vector.unsqueeze(
            1).repeat(1, enc_src.size(1), 1)

        enc_src = enc_src + truth_table_vector

        output, attention = self.decoder(trg, enc_src, trg_mask, src_mask)
        # output = [batch size, trg len, output dim]
        # attention = [batch size, n heads, trg len, src len]

        return output, attention

## train/evaluate

In [5]:
def train(model, iterator, optimizer, criterion, clip, device):
    model.train()
    epoch_loss = 0
    total_correct = 0
    total_count = 0

    pbar = tqdm(iterator, unit='batchs', dynamic_ncols=True,
                leave=False, position=0, desc="Train", colour='green')

    for i, batch in enumerate(pbar):

        # Initialize gradients
        optimizer.zero_grad()

        # Move data to GPU
        src = batch.src.to(device)
        trg = batch.trg.to(device)

        truth_table = batch.truth_table.float().to(device)

        # Forward propagation
        output, _ = model(src, trg[:, :-1], truth_table)
        # output = [batch size, trg len - 1, output dim]
        # trg = [batch size, trg len]

        output_dim = output.shape[-1]

        output = output.contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)
        # output = [batch size * trg len - 1, output dim]
        # trg = [batch size * trg len - 1]

        # Loss calculation
        loss = criterion(output, trg)

        # Backward propagation
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        # Gradient descent
        optimizer.step()

        # Accumulate loss
        epoch_loss += loss.item()

        # Accuracy calculation
        # Get the index of the max log-probability (prediction)
        pred = output.argmax(1)
        # Count how many predictions are correct
        correct = (pred == trg).float().sum()
        total_correct += correct
        # Total number of elements to compare
        total_count += trg.size(0)

        # Update progress bar
        pbar.set_postfix(loss="{:.04f}".format(float(epoch_loss / (i + 1))),
                         acc="{:.04f}%".format(float(total_correct * 100 / total_count)))
        pbar.update()

    pbar.close()

    # Release memory

    del src, trg, truth_table, output
    torch.cuda.empty_cache()

    # Calculate final average epoch_loss
    epoch_loss = epoch_loss / len(iterator)

    # Calculate final accuracy as a percentage
    epoch_acc = total_correct / total_count

    return epoch_loss, epoch_acc

In [6]:
def evaluate(model, iterator, criterion):
    model.eval()

    epoch_loss = 0
    total_correct = 0
    total_count = 0

    pbar = tqdm(iterator, unit='batchs', dynamic_ncols=True,
                leave=False, position=0, desc="Valid", colour='cyan')

    with torch.no_grad():
        for i, batch in enumerate(pbar):

            # Move data to GPU
            src = batch.src
            trg = batch.trg
            truth_table = batch.truth_table

            output, _ = model(src, trg[:, :-1], truth_table)
            # output = [batch size, trg len - 1, output dim]
            # trg = [batch size, trg len]

            output_dim = output.shape[-1]

            output = output.contiguous().view(-1, output_dim)
            trg = trg[:, 1:].contiguous().view(-1)
            # output = [batch size * trg len - 1, output dim]
            # trg = [batch size * trg len - 1]

            loss = criterion(output, trg)

            # Accumulate loss
            epoch_loss += loss.item()

            # Accuracy calculation
            # Get the index of the max log-probability (prediction)
            pred = output.argmax(1)
            correct = (pred == trg).float().sum()   # Count correct predictions
            total_correct += correct
            # Total number of elements to compare
            total_count += trg.size(0)

            # Update progress bar
            pbar.set_postfix(loss="{:.04f}".format(float(epoch_loss / (i + 1))),
                             acc="{:.04f}%".format(float(total_correct * 100 / total_count)))
            pbar.update()

    pbar.close()

    # Release memory
    del src, trg, truth_table, output
    torch.cuda.empty_cache()

    # Calculate loss final average loss
    epoch_loss = epoch_loss / len(iterator)

    # Calculate accuracy as a percentage
    epoch_acc = total_correct / total_count

    return epoch_loss, epoch_acc

## translate/count_acc

In [7]:
def translate(sentence, src_field, trg_field, model, device, truth_table, max_len=100):
    model.eval()

    tokens = [token.lower() for token in sentence]
    tokens = [src_field.init_token] + tokens + [src_field.eos_token]

    src_indexes = [src_field.vocab.stoi[token] for token in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(0).to(device)
    src_mask = model.make_src_mask(src_tensor)

    with torch.no_grad():

        enc_src = model.encoder(src_tensor, src_mask)
        truth_table_vector = model.truth_table_linear(truth_table)
        truth_table_vector = truth_table_vector.unsqueeze(
            1).repeat(1, enc_src.size(1), 1)

        enc_src = enc_src + truth_table_vector

    trg_indexes = [trg_field.vocab.stoi[trg_field.init_token]]

    for i in range(max_len):

        trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(0).to(device)
        trg_mask = model.make_trg_mask(trg_tensor)

        with torch.no_grad():
            output, attention = model.decoder(
                trg_tensor, enc_src, trg_mask, src_mask)

        pred_token = output.argmax(2)[:, -1].item()
        trg_indexes.append(pred_token)

        if pred_token == trg_field.vocab.stoi[trg_field.eos_token]:
            break

    trg_tokens = [trg_field.vocab.itos[i] for i in trg_indexes]

    return trg_tokens[1:]

In [8]:
def count_acc(dataset, SRC, TRG, model, device):
    count = 0

    # for idx in tqdm(len(dataset), ncols=100, leave=True, colour="yellow", desc="Final Valid"):
    for idx in tqdm(range(min(10000, len(dataset))), dynamic_ncols=True, desc="Final Valid", colour="yellow"):
        src = vars(dataset.examples[idx])['src']
        trg = vars(dataset.examples[idx])['trg']
        truth_table = torch.Tensor(vars(dataset.examples[idx])[
                                   'truth_table']).unsqueeze(0).to(device)

        translation = translate(src, SRC, TRG, model, device, truth_table)

        if translation[:-1] == trg:
            count += 1
    return count

## Train

### Preparation

In [9]:
# 실험 재현을 위한 seed 고정

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [10]:
SRC = Field(
    tokenize=_tokenize,
    init_token='<sos>',
    eos_token='<eos>',
    pad_token='<pad>',
    lower=True,
    batch_first=True,
)
TRG = Field(
    tokenize=_tokenize,
    init_token='<sos>',
    eos_token='<eos>',
    pad_token='<pad>',
    lower=True,
    batch_first=True
)
TRUTH_TABLE = Field(
    use_vocab=False,
    dtype=torch.float32,
    preprocessing=lambda x: [int(num.strip('[],')) for num in x],
    batch_first=True
)

SRC, TRG, TRUTH_TABLE

(<torchtext.data.field.Field at 0x231a0150610>,
 <torchtext.data.field.Field at 0x2319fae5e90>)

In [11]:
exprs = torchtext.data.TabularDataset(
    path='../data/train_s-bool_tt_added.csv',
    format='csv',
    fields=[
        ('src', SRC),
        ('trg', TRG),
        ("truth_table", TRUTH_TABLE)
    ]
)
train_data, valid_data = exprs.split(split_ratio=0.8)

train_data, valid_data

(<torchtext.data.dataset.Dataset at 0x2319d9ca2d0>,
 <torchtext.data.dataset.Dataset at 0x231a0155fd0>)

In [12]:
print(f'Total {len(exprs)} samples.')
print(f'Total {len(train_data)} train samples.')
print(f'Total {len(valid_data)} valid samples.')
print()

print('example expression pair')
print(*exprs.examples[-1].src, sep='')
print(*exprs.examples[-1].trg, sep='')
print()

# Build vocab only from the training set, which can prevent information leakage
SRC.build_vocab(train_data)
print("src vocab size: {}".format(len(SRC.vocab)))
print(SRC.vocab.stoi)

TRG.build_vocab(train_data)
print("trg vocab size: {}".format(len(TRG.vocab)))
print(TRG.vocab.stoi)
print()

print(f'Total {len(SRC.vocab)} unique tokens in source vocabulary')
print(f'Total {len(TRG.vocab)} unique tokens in target vocabulary')

Total 100000 samples.
Total 80000 train samples.
Total 20000 valid samples.

example expression pair
(x|y)+2*(x^y)+6*(~(x|y))-2*(~y)-(x|~y)-(~x)-(~x|y)-(~(x&y))
-x

src vocab size: 28
defaultdict(<bound method Vocab._default_unk_index of <torchtext.vocab.Vocab object at 0x000002319DF62B90>>, {'<unk>': 0, '<pad>': 1, '<sos>': 2, '<eos>': 3, '(': 4, ')': 5, '~': 6, 'y': 7, 'x': 8, '-': 9, 'z': 10, '&': 11, '*': 12, '|': 13, '+': 14, '^': 15, '2': 16, '3': 17, '4': 18, '1': 19, '5': 20, '6': 21, 'a': 22, '7': 23, 'b': 24, '8': 25, '9': 26, '0': 27})
trg vocab size: 28
defaultdict(<bound method Vocab._default_unk_index of <torchtext.vocab.Vocab object at 0x000002319F9CC5D0>>, {'<unk>': 0, '<pad>': 1, '<sos>': 2, '<eos>': 3, '(': 4, ')': 5, 'x': 6, 'y': 7, '-': 8, '&': 9, '*': 10, '~': 11, '^': 12, 'z': 13, '+': 14, '1': 15, '2': 16, '|': 17, '3': 18, '4': 19, 'a': 20, '5': 21, 'b': 22, '6': 23, '7': 24, '8': 25, '9': 26, '0': 27})

Total 28 unique tokens in source vocabulary
Total 28 uniqu

### Parameter Configuration

In [13]:
config = {
    'epochs': 1,
    'batch_size': 1024,

    'HID_DIM': 256,
    'ENC_LAYERS': 3,
    'DEC_LAYERS': 3,
    'ENC_HEADS': 8,
    'DEC_HEADS': 8,
    'ENC_PF_DIM': 512,
    'DEC_PF_DIM': 512,

    'ENC_DROPOUT': 0.1,
    'DEC_DROPOUT': 0.1,

    'MAX_LENGTH': 104,
}

### Preparation2

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_DIM = len(SRC.vocab)
OUTPUT_DIM = len(TRG.vocab)

print(INPUT_DIM, OUTPUT_DIM)

28 28


In [15]:
enc = Encoder(INPUT_DIM,
              config['HID_DIM'],
              config['ENC_LAYERS'],
              config['ENC_HEADS'],
              config['ENC_PF_DIM'],
              config['ENC_DROPOUT'],

              device=device,
              max_length=config['MAX_LENGTH']
              )

In [16]:
dec = Decoder(OUTPUT_DIM,
              config['HID_DIM'],
              config['DEC_LAYERS'],
              config['DEC_HEADS'],
              config['DEC_PF_DIM'],
              config['DEC_DROPOUT'],

              device=device,
              max_length=config['MAX_LENGTH']
              )

In [17]:
SRC_PAD_IDX = SRC.vocab.stoi[SRC.pad_token]
TRG_PAD_IDX = TRG.vocab.stoi[TRG.pad_token]

SRC_PAD_IDX, TRG_PAD_IDX

(1, 1)

In [18]:
train_iter, valid_iter = BucketIterator.splits(
    (train_data, valid_data),
    batch_size=config['batch_size'],

    sort=False,
    device=device
)

In [19]:
today = datetime.date.today()

model = Seq2Seq(enc, dec, SRC_PAD_IDX, TRG_PAD_IDX, device).to(device)
print(f'The model has {count_parameters(model):,} trainable parameters')

model.apply(initialize_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer)
criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)

The model has 4,036,892 trainable parameters


### Experiment

In [20]:
torch.cuda.empty_cache()
gc.collect()
# wandb.watch(model, log="all")

best_valid_loss = float('inf')

for epoch in range(config["epochs"]):

    print("\nEpoch {}/{}".format(epoch+1, config['epochs']))

    curr_lr = float(optimizer.param_groups[0]["lr"])
    train_loss, train_acc = train(
        model, train_iter, optimizer, criterion, clip=1, device=device)
    valid_loss, valid_acc = evaluate(model, valid_iter, criterion)

    print("\tTrain Acc {:.04f}%\tTrain Loss {:.04f}\t Learning Rate {:.07f}".format(
        train_acc*100, train_loss, curr_lr))
    print("\tValid Acc {:.04f}%\tValid Loss {:.04f}".format(
        valid_loss*100, valid_acc))

    # wandb.log({'train_acc': train_acc*100,      'train_loss': train_loss,
    #            'valid_acc': valid_acc*100,      'valid_loss': valid_loss,   'lr': curr_lr})

    scheduler.step(valid_loss)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(
        ), f'../checkpoints/transformer_add_bool_tt-{today}-{config["epochs"]}epochs.pt')

    print(
        f'\nEpochs: {epoch + 1}, Train Loss: {train_loss:.3f}, Valid Loss: {valid_loss:.3f}\n')


# valid_acc_count = count_acc(valid_data, SRC, TRG, model, device)
# print(f'\nFinal Accuracy rate on valid set: {valid_acc_count / len(valid_data):.3f}, count: {valid_acc_count:>5d}/{len(valid_data):>5d}')


Epoch 1/1


                                                                                     s=0.4426]

	Train Acc 32.5374%	Train Loss 1.0817	 Learning Rate 0.0005000
	Valid Acc 44.4140%	Valid Loss 0.4121

Epochs: 1, Train Loss: 1.082, Valid Loss: 0.444



## Test

### test

In [21]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def evaluate_metrics(test_data, SRC, TRG, model, device):
    model.eval()
    total_correct = 0
    total_count = 0
    total_bleu_score = 0
    smoothie = SmoothingFunction().method4

    pbar = tqdm(test_data.examples, colour="blue", desc="Test")

    with torch.no_grad():
        for example in pbar:
            # Extract data from the example
            src = vars(example)['src']
            trg = vars(example)['trg']
            truth_table = torch.tensor(
                vars(example)['truth_table']).float().unsqueeze(0).to(device)

            # Perform inference once
            prediction = translate(src, SRC, TRG, model, device, truth_table)[
                :-1]  # Ignore <eos>

            # Calculate accuracy
            if prediction == trg:
                total_correct += 1
            total_count += 1

            # Calculate BLEU score
            total_bleu_score += sentence_bleu([trg],
                                              prediction, smoothing_function=smoothie)

    accuracy = total_correct / total_count
    bleu = total_bleu_score / total_count

    return accuracy, bleu

### evaluate with metric

In [23]:
# Load test data
test_data = torchtext.data.TabularDataset(
    path='../data/test-bool_tt_added.csv',
    format='csv',
    fields=[('src', SRC), ('trg', TRG), ("truth_table", TRUTH_TABLE)]
)

In [24]:
torch.cuda.empty_cache()
gc.collect()

# project_name = "Test_transformer_100K_arch_changed(concat)-0901"
# experiment_name = "test_evaluation-0902-100epochs"
# wandb.init(project=project_name, name=experiment_name)

# =================== Load the saved model ===================
# model_path = './checkpoints/100K/changed/transformer2024-09-09-20epochs.pt'
# ============================================================

# model_4_test = Seq2Seq(enc, dec, SRC_PAD_IDX, TRG_PAD_IDX, device).to(device)
# model_4_test.load_state_dict(torch.load(model_path, map_location=device), strict=False)


# Calculate accuracy and BLEU together
accuracy, bleu = evaluate_metrics(test_data, SRC, TRG, model, device)
print(f'Test Accuracy: {accuracy:.4f}')
print(f'Test BLEU: {bleu:.4f}')

Test: 100%|██████████| 10000/10000 [07:54<00:00, 21.09it/s]

Test Accuracy: 0.0233
Test BLEU: 0.4474
